# KYC — Étape 2.1 : Détection d'objets sur documents d'identité

**Modèle :** YOLOv11s (Ultralytics) · **Dataset :** MIDV-2020 (photo split) · **Compute :** Colab T4 (free)

Ce notebook produit un détecteur d'objets capable de localiser sur un document d'identité :
- la **photo** du porteur (input pour l'étape 2.2 face-match)
- la **date d'expiration** (champ critique pour le KYC)
- la **MRZ**, le **nom**, la **date de naissance**, le **numéro de document**

> 📌 Lis d'abord `plan_technique_kyc_etape_2_1.md` pour le rationale complet (choix dataset, choix modèle, stratégie d'entraînement).

---

**Exécution :** *Runtime → Change runtime type → T4 GPU* avant de lancer les cellules.


## 1. Setup & vérification GPU

In [ ]:
# Install Ultralytics (YOLOv11) + utilitaires
!pip install -q ultralytics==8.3.40 roboflow gdown opencv-python-headless

import torch, os, sys
print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")
print(f"CUDA dispo : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ Pas de GPU détecté. Va dans Runtime → Change runtime type → T4 GPU")


In [ ]:
# Structure du workspace
import pathlib

WORKDIR = pathlib.Path("/content/kyc_2_1")
WORKDIR.mkdir(exist_ok=True)
(WORKDIR / "raw").mkdir(exist_ok=True)        # dataset téléchargé
(WORKDIR / "yolo").mkdir(exist_ok=True)        # dataset au format YOLO
(WORKDIR / "yolo" / "images").mkdir(exist_ok=True)
(WORKDIR / "yolo" / "labels").mkdir(exist_ok=True)
for split in ("train", "val", "test"):
    (WORKDIR / "yolo" / "images" / split).mkdir(exist_ok=True)
    (WORKDIR / "yolo" / "labels" / split).mkdir(exist_ok=True)

print(f"Workspace : {WORKDIR}")
!ls -la {WORKDIR}


## 2. Téléchargement de MIDV-2020 (photo split)

### Pourquoi MIDV-2020 et pas MIDV-500 ?

Le repo `fcakyon/midv500` (référencé dans l'énoncé) ne fournit que les **annotations des 4 coins du document**, pas les champs internes (photo, dates...). MIDV-2020 est l'évolution officielle de la même famille, avec des annotations **field-level** au format VIA JSON.

→ On télécharge le split `photo` (1000 images de documents photographiés au smartphone, conditions réalistes pour KYC mobile).

### 2 voies au choix :

- **Voie A (recommandée) :** dataset officiel depuis l3i-share / Smart Engines FTP
- **Voie B (express) :** version pré-formatée YOLO sur Roboflow Universe — gain de temps mais classes/splits fixés par un tiers

Décommente la voie que tu choisis ci-dessous.


In [ ]:
# ===== VOIE A : MIDV-2020 officiel =====
# Le dataset est hébergé sur ftp://smartengines.com/midv-2020 et l3i-share.univ-lr.fr
# Lien direct du split photo (mirror HTTPS si dispo) :
MIDV2020_PHOTO_URL = "ftp://smartengines.com/midv-2020/photo.tar"

# Téléchargement (curl supporte FTP)
import subprocess, os
RAW = WORKDIR / "raw"

if not (RAW / "photo.tar").exists():
    print("⏬ Téléchargement MIDV-2020 photo (~2 GB)...")
    ret = subprocess.run(
        ["curl", "-L", "-o", str(RAW / "photo.tar"), MIDV2020_PHOTO_URL],
        capture_output=True, text=True
    )
    if ret.returncode != 0:
        print("⚠️ Échec FTP. Essaie le mirror :")
        print("    https://l3i-share.univ-lr.fr/MIDV2020/")
        print("    (accès via formulaire, télécharge `photo.tar` manuellement et upload dans /content/kyc_2_1/raw/)")
else:
    print(f"✅ Déjà téléchargé : {RAW / 'photo.tar'}")

# Extraction
if (RAW / "photo.tar").exists() and not (RAW / "photo").exists():
    print("📂 Extraction...")
    !tar -xf {RAW}/photo.tar -C {RAW}/
    print("✅ Extrait dans", RAW / "photo")


In [ ]:
# ===== VOIE B : Roboflow (alternative express) =====
# Décommente si la voie A est trop lente ou bloquée.
# Tu auras besoin d'une clé API gratuite : https://app.roboflow.com/settings/api
#
# from roboflow import Roboflow
# rf = Roboflow(api_key="VOTRE_CLE_ICI")
# project = rf.workspace("maastricht-university").project("genmrp-midv-2020")
# dataset = project.version(1).download("yolov8", location=str(WORKDIR / "yolo_roboflow"))
# print("✅ Dataset YOLO prêt :", dataset.location)
#
# → Si tu utilises cette voie, saute les sections 3 et 4 (conversion déjà faite)
#   et passe directement à la section 6 en ajustant data.yaml sur dataset.location/data.yaml


## 3. Inspection des annotations VIA

MIDV-2020 utilise le format **VIA v2** (VGG Image Annotator) : un JSON par sous-ensemble qui contient pour chaque image une liste de "régions" avec :
- `shape_attributes` : géométrie (polygone, généralement quadrilatère)
- `region_attributes` : attributs sémantiques (le label du champ)

On parcourt un fichier d'annotation pour voir les labels disponibles.


In [ ]:
import json
from pathlib import Path
from collections import Counter

# Recherche d'un fichier d'annotation VIA dans le dataset extrait
RAW = WORKDIR / "raw"
ann_files = list(RAW.rglob("*.json"))
print(f"Trouvé {len(ann_files)} fichiers JSON.")
for f in ann_files[:5]:
    print(" -", f.relative_to(RAW))

# On ouvre le premier pour voir la structure
if ann_files:
    with open(ann_files[0]) as f:
        via = json.load(f)
    # Structure VIA v2 : { "_via_img_metadata": { "image_key": { "filename", "regions": [...] } } }
    # OU directement { "image_key": {...} } selon la version
    if "_via_img_metadata" in via:
        metadata = via["_via_img_metadata"]
    else:
        metadata = via
    print(f"\n{len(metadata)} images annotées dans ce fichier.")

    # Liste les labels distincts
    label_counter = Counter()
    for img_key, img_data in metadata.items():
        for region in img_data.get("regions", []):
            attrs = region.get("region_attributes", {})
            # Le label peut être sous différentes clés selon la version
            label = attrs.get("field_name") or attrs.get("class") or attrs.get("label") or str(attrs)
            label_counter[label] += 1

    print("\n📊 Top 20 labels :")
    for lab, n in label_counter.most_common(20):
        print(f"  {n:5d}  {lab}")
else:
    print("⚠️ Aucun JSON trouvé. Vérifie l'extraction (section 2).")


## 4. Conversion VIA → format YOLO

### Format cible YOLO
Un fichier `.txt` par image, **même nom** que l'image, **même répertoire que `labels/<split>/`**. Une ligne par bbox :
```
class_id  x_center_norm  y_center_norm  width_norm  height_norm
```
Toutes les valeurs ∈ [0, 1], normalisées par les dimensions de l'image.

### Notre mapping de classes

| ID | Label MIDV (variantes possibles) | Notre classe |
|---|---|---|
| 0 | `photo`, `portrait`, `face` | `photo` |
| 1 | `mrz`, `mrz_line_1`, `mrz_line_2` (mergés) | `mrz` |
| 2 | `surname`, `name`, `given_names` | `name` |
| 3 | `date_of_birth`, `birth_date`, `dob` | `birth_date` |
| 4 | `date_of_expiry`, `expiry_date`, `expiration` | `expiry_date` |
| 5 | `document_number`, `doc_number`, `passport_number`, `id_number` | `document_number` |

Tous les autres labels (signature, country, sex, etc.) sont ignorés pour rester focused sur les 6 classes prioritaires.


In [ ]:
# ===== Mapping label MIDV → classe YOLO =====
# Plusieurs variantes par classe car MIDV-2020 a des conventions de nommage qui varient selon le type de doc.
CLASS_MAPPING = {
    # photo
    "photo": 0, "portrait": 0, "face": 0, "photograph": 0,
    # mrz (toutes les lignes mergées en une classe)
    "mrz": 1, "mrz_line_1": 1, "mrz_line_2": 1, "mrz_line_3": 1, "machine_readable_zone": 1,
    # name
    "surname": 2, "name": 2, "given_names": 2, "given_name": 2, "first_name": 2, "last_name": 2,
    # birth date
    "date_of_birth": 3, "birth_date": 3, "dob": 3, "birthdate": 3,
    # expiry date
    "date_of_expiry": 4, "expiry_date": 4, "expiration": 4, "date_of_expiration": 4,
    # document number
    "document_number": 5, "doc_number": 5, "passport_number": 5, "id_number": 5,
    "card_number": 5, "number": 5,
}

CLASS_NAMES = ["photo", "mrz", "name", "birth_date", "expiry_date", "document_number"]
NUM_CLASSES = len(CLASS_NAMES)
print(f"{NUM_CLASSES} classes : {CLASS_NAMES}")


In [ ]:
from PIL import Image
import shutil

def polygon_to_yolo_bbox(points_x, points_y, img_w, img_h):
    """Convertit un polygone (liste de x, liste de y) en bbox YOLO normalisée."""
    x_min, x_max = min(points_x), max(points_x)
    y_min, y_max = min(points_y), max(points_y)
    cx = (x_min + x_max) / 2 / img_w
    cy = (y_min + y_max) / 2 / img_h
    w = (x_max - x_min) / img_w
    h = (y_max - y_min) / img_h
    # Clamp [0, 1] (anti-débordements liés à l'annotation)
    return (
        max(0.0, min(1.0, cx)),
        max(0.0, min(1.0, cy)),
        max(0.0, min(1.0, w)),
        max(0.0, min(1.0, h)),
    )

def extract_label(region_attrs):
    """Récupère le label d'une région VIA (clés variables)."""
    for key in ("field_name", "class", "label", "type"):
        if key in region_attrs:
            return str(region_attrs[key]).lower().strip()
    return None

def convert_via_to_yolo(via_json_path, images_root, out_images_dir, out_labels_dir, mapping):
    """Lit un JSON VIA, convertit, copie images et écrit labels YOLO."""
    with open(via_json_path) as f:
        via = json.load(f)
    metadata = via.get("_via_img_metadata", via)

    n_imgs_ok, n_imgs_skip, n_boxes = 0, 0, 0
    images_root = Path(images_root)
    out_images_dir = Path(out_images_dir)
    out_labels_dir = Path(out_labels_dir)

    for img_key, img_data in metadata.items():
        filename = img_data.get("filename")
        if not filename:
            continue
        src_img = images_root / filename
        if not src_img.exists():
            # Cherche récursivement (les images peuvent être dans des sous-dossiers)
            matches = list(images_root.rglob(filename))
            if not matches:
                n_imgs_skip += 1
                continue
            src_img = matches[0]

        # Lis les dimensions
        try:
            with Image.open(src_img) as im:
                img_w, img_h = im.size
        except Exception:
            n_imgs_skip += 1
            continue

        lines = []
        for region in img_data.get("regions", []):
            label = extract_label(region.get("region_attributes", {}))
            if label not in mapping:
                continue
            class_id = mapping[label]
            shape = region.get("shape_attributes", {})
            # 3 formes possibles : polygon, polyline, rect
            if shape.get("name") in ("polygon", "polyline"):
                px = shape.get("all_points_x", [])
                py = shape.get("all_points_y", [])
            elif shape.get("name") == "rect":
                x, y, w, h = shape["x"], shape["y"], shape["width"], shape["height"]
                px = [x, x + w, x + w, x]
                py = [y, y, y + h, y + h]
            else:
                continue
            if len(px) < 2 or len(py) < 2:
                continue
            cx, cy, bw, bh = polygon_to_yolo_bbox(px, py, img_w, img_h)
            if bw <= 0 or bh <= 0:
                continue
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            n_boxes += 1

        if not lines:
            n_imgs_skip += 1
            continue

        # Copie l'image + écrit le label (nom unique pour éviter les collisions inter-dossiers)
        unique_name = src_img.stem + "_" + str(hash(str(src_img)) & 0xFFFFFF) + src_img.suffix
        dst_img = out_images_dir / unique_name
        dst_lbl = out_labels_dir / (Path(unique_name).stem + ".txt")
        shutil.copy(src_img, dst_img)
        dst_lbl.write_text("\n".join(lines))
        n_imgs_ok += 1

    return n_imgs_ok, n_imgs_skip, n_boxes


# Stockage temporaire avant split — on convertit tout dans un même bucket, on split après
TMP_IMG = WORKDIR / "yolo_all" / "images"
TMP_LBL = WORKDIR / "yolo_all" / "labels"
TMP_IMG.mkdir(parents=True, exist_ok=True)
TMP_LBL.mkdir(parents=True, exist_ok=True)

total_ok, total_skip, total_boxes = 0, 0, 0
for ann_file in ann_files:
    # Heuristique : le dossier images est au même niveau que l'annotation
    images_root = ann_file.parent
    ok, skip, boxes = convert_via_to_yolo(ann_file, images_root, TMP_IMG, TMP_LBL, CLASS_MAPPING)
    total_ok += ok
    total_skip += skip
    total_boxes += boxes

print(f"\n✅ Conversion terminée :")
print(f"   {total_ok} images converties")
print(f"   {total_skip} images skipped (sans labels mappables ou non trouvées)")
print(f"   {total_boxes} bounding boxes au total")
print(f"   Moyenne : {total_boxes / max(1, total_ok):.1f} bboxes / image")


## 5. Split train / val / test (70 / 20 / 10)

On split aléatoirement (random seed fixé pour reproductibilité). MIDV-2020 photo contient 1000 images relativement homogènes, donc un random split est OK ; pas besoin de stratification complexe pour ce volume.


In [ ]:
import random

random.seed(42)
all_images = sorted([p.name for p in TMP_IMG.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".tif", ".tiff")])
random.shuffle(all_images)

n = len(all_images)
n_train = int(0.7 * n)
n_val = int(0.2 * n)
splits = {
    "train": all_images[:n_train],
    "val":   all_images[n_train:n_train + n_val],
    "test":  all_images[n_train + n_val:],
}

YOLO_ROOT = WORKDIR / "yolo"
for split, names in splits.items():
    for name in names:
        src_img = TMP_IMG / name
        src_lbl = TMP_LBL / (Path(name).stem + ".txt")
        shutil.move(src_img, YOLO_ROOT / "images" / split / name)
        if src_lbl.exists():
            shutil.move(src_lbl, YOLO_ROOT / "labels" / split / src_lbl.name)
    print(f"{split:6s} : {len(names)} images")


## 6. Fichier `data.yaml` (config Ultralytics)

In [ ]:
DATA_YAML = WORKDIR / "data.yaml"
yaml_content = f"""# YOLOv11 dataset config — KYC step 2.1
path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test

nc: {NUM_CLASSES}
names:
{chr(10).join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))}
"""
DATA_YAML.write_text(yaml_content)
print(yaml_content)


## 7. Entraînement de YOLOv11s

Paramètres importants :
- **`fliplr=0.0`** : on désactive le flip horizontal (les dates/MRZ ne sont pas symétriques)
- **`close_mosaic=10`** : on désactive le mosaic sur les 10 dernières epochs (stabilise la fin de training)
- **`patience=15`** : early stopping si pas d'amélioration sur 15 epochs


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")  # weights COCO pré-entraînés

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=15,
    close_mosaic=10,
    # Augmentations spécifiques documents
    fliplr=0.0,             # IMPORTANT : pas de flip horizontal
    flipud=0.0,             # idem vertical
    degrees=10.0,           # rotations légères OK
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mixup=0.0,              # à augmenter à 0.1 si overfit
    # Sortie
    project=str(WORKDIR / "runs"),
    name="kyc_2_1_yolov11s",
    exist_ok=True,
    save=True,
    plots=True,
    verbose=True,
)

BEST_MODEL = WORKDIR / "runs" / "kyc_2_1_yolov11s" / "weights" / "best.pt"
print(f"\n✅ Best model : {BEST_MODEL}")


## 8. Évaluation sur val + test

On regarde la mAP par classe — la classe `photo` doit être >0.95 (elle est facile à apprendre, et c'est la classe critique pour l'étape 2.2).


In [ ]:
# Eval sur le val set (auto pendant le training, on récupère les chiffres)
model = YOLO(str(BEST_MODEL))
val_metrics = model.val(data=str(DATA_YAML), split="val", plots=True, save_json=True)

print("\n=== Val metrics globales ===")
print(f"mAP@50     : {val_metrics.box.map50:.4f}")
print(f"mAP@50-95  : {val_metrics.box.map:.4f}")
print(f"Precision  : {val_metrics.box.mp:.4f}")
print(f"Recall     : {val_metrics.box.mr:.4f}")

print("\n=== Par classe (mAP@50) ===")
for i, name in enumerate(CLASS_NAMES):
    if i < len(val_metrics.box.maps):
        ap50 = val_metrics.box.ap50[i] if hasattr(val_metrics.box, 'ap50') else val_metrics.box.maps[i]
        print(f"  {name:18s} : {ap50:.4f}")


In [ ]:
# Eval sur le test set (jamais vu pendant le training)
test_metrics = model.val(data=str(DATA_YAML), split="test", plots=True)
print(f"\n=== Test set ===")
print(f"mAP@50     : {test_metrics.box.map50:.4f}")
print(f"mAP@50-95  : {test_metrics.box.map:.4f}")


In [ ]:
# Visualisation des artefacts générés par Ultralytics
from IPython.display import Image as IPImage, display

run_dir = WORKDIR / "runs" / "kyc_2_1_yolov11s"
for fname in ("confusion_matrix.png", "results.png", "F1_curve.png", "PR_curve.png"):
    fpath = run_dir / fname
    if fpath.exists():
        print(f"\n=== {fname} ===")
        display(IPImage(str(fpath)))


## 9. Inférence sur quelques images du test set

On vérifie visuellement la qualité des détections.


In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

test_imgs = sorted((YOLO_ROOT / "images" / "test").iterdir())[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flat, test_imgs):
    results = model.predict(str(img_path), conf=0.25, verbose=False)
    plotted = results[0].plot()  # numpy BGR avec bboxes
    plotted_rgb = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
    ax.imshow(plotted_rgb)
    ax.axis('off')
    ax.set_title(img_path.name[:25], fontsize=9)
plt.tight_layout()
plt.show()


## 10. Interface de production — `predict_and_crop()`

Cette fonction est **le pont vers l'étape 2.2 et l'étape 3** :
- Elle prend une image en entrée
- Elle retourne un dict structuré avec, pour chaque classe détectée : bbox, confiance, et chemin vers le crop sauvegardé
- Elle flag les cas problématiques (photo manquante, faible confiance)

C'est cette fonction que tu vas exposer à tes coéquipiers (ou réutiliser pour l'étape 3).


In [ ]:
from pathlib import Path
import cv2
import json as _json

def predict_and_crop(image_path, output_dir="/content/kyc_2_1/crops",
                     conf_threshold=0.25, model_obj=None):
    """
    Détecte tous les champs sur un document d'identité et sauvegarde les crops.

    Returns
    -------
    dict avec :
      - image_path : chemin original
      - detections : dict {class_name: {bbox, conf, crop_path}}
      - missing_fields : liste des classes attendues mais non détectées
      - is_valid_kyc_input : booléen (au minimum photo détectée avec conf > 0.7)
    """
    mdl = model_obj if model_obj is not None else model
    image_path = Path(image_path)
    output_dir = Path(output_dir) / image_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)

    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        return {"error": "Impossible de lire l'image", "image_path": str(image_path)}
    h, w = img_bgr.shape[:2]

    results = mdl.predict(str(image_path), conf=conf_threshold, verbose=False)[0]

    detections = {}
    for box in results.boxes:
        cls_id = int(box.cls.item())
        cls_name = CLASS_NAMES[cls_id]
        conf = float(box.conf.item())
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        # Si une classe apparaît plusieurs fois, on garde la plus confiante
        if cls_name in detections and detections[cls_name]["conf"] >= conf:
            continue

        # Sauvegarde du crop
        crop = img_bgr[y1:y2, x1:x2]
        crop_path = output_dir / f"{cls_name}.png"
        cv2.imwrite(str(crop_path), crop)

        detections[cls_name] = {
            "bbox": [x1, y1, x2, y2],
            "conf": round(conf, 4),
            "crop_path": str(crop_path),
        }

    missing = [c for c in CLASS_NAMES if c not in detections]
    is_valid = ("photo" in detections) and (detections["photo"]["conf"] > 0.7)

    return {
        "image_path": str(image_path),
        "detections": detections,
        "missing_fields": missing,
        "is_valid_kyc_input": is_valid,
    }

# Test sur une image
test_image = test_imgs[0]
result = predict_and_crop(test_image)
print(_json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
# Inspecte les crops produits (en particulier le crop photo qui ira en input de l'étape 2.2)
from IPython.display import Image as IPImage, display

if "photo" in result["detections"]:
    print("📸 Crop photo (input étape 2.2 face-match) :")
    display(IPImage(result["detections"]["photo"]["crop_path"]))
else:
    print("⚠️ Pas de photo détectée sur cette image — input invalide pour étape 2.2")


## 11. Règles de validation KYC embarquées

Quelques règles métier pour flag les inputs douteux avant de les envoyer aux étapes suivantes.


In [ ]:
def validate_detection_quality(prediction_result):
    """Renvoie une liste de warnings et un statut global."""
    warnings_list = []
    status = "OK"

    det = prediction_result.get("detections", {})
    if not det:
        return {"status": "REJECTED", "warnings": ["Aucune détection — image probablement invalide"]}

    # Photo obligatoire
    if "photo" not in det:
        warnings_list.append("CRITIQUE : photo du porteur non détectée")
        status = "REJECTED"
    elif det["photo"]["conf"] < 0.7:
        warnings_list.append(f"Photo détectée avec faible confiance ({det['photo']['conf']:.2f})")
        status = "REVIEW"

    # Confiance moyenne
    if det:
        mean_conf = sum(d["conf"] for d in det.values()) / len(det)
        if mean_conf < 0.7:
            warnings_list.append(f"Confiance moyenne faible ({mean_conf:.2f}) — doc potentiellement flou")
            status = "REVIEW" if status == "OK" else status

    # Champs manquants
    if len(prediction_result["missing_fields"]) > 3:
        warnings_list.append(f"{len(prediction_result['missing_fields'])} champs manquants — doc tronqué ?")
        status = "REVIEW" if status == "OK" else status

    return {"status": status, "warnings": warnings_list}


# Test
quality = validate_detection_quality(result)
print(_json.dumps(quality, indent=2, ensure_ascii=False))


## 12. Export du modèle pour déploiement

Format ONNX = portable (CPU, mobile, edge). Format TorchScript = production PyTorch.


In [ ]:
# ONNX (pour serving CPU / mobile)
model.export(format="onnx", imgsz=640, opset=12)

# TorchScript (pour serving PyTorch)
model.export(format="torchscript", imgsz=640)

# Liste les fichiers exportés
exported_dir = BEST_MODEL.parent
for f in exported_dir.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# Téléchargement local (Colab) — décommenter pour download dans le navigateur
# from google.colab import files
# files.download(str(BEST_MODEL))
# files.download(str(BEST_MODEL.with_suffix('.onnx')))


## 13. Pour aller plus loin (bonus soutenance)

### Pipeline 2 étapes (gros gain de mAP attendu)

L'idée : MIDV-500 sait localiser le document (corners), MIDV-2020 sait localiser les fields. On peut les chaîner :

1. **YOLOv11 #1** entraîné sur MIDV-500 → détecte les 4 coins du document
2. **Homographie** → rectifie le document (vue de dessus, taille standardisée)
3. **YOLOv11 #2** (celui de ce notebook) → détecte les fields sur l'image rectifiée

Gain attendu : +5 à +8 pts mAP, parce que le modèle field-level voit toujours le document dans la même orientation/résolution.

### Test-Time Augmentation (gratuit, +1-2 pts)

```python
model.val(data=str(DATA_YAML), augment=True)
```

### Données externes

MIDV est mock (documents fictifs). En production il faut compléter avec un dataset privé contenant des CNI/passeports FR. Cf. annotation via [Roboflow Annotate](https://roboflow.com/annotate) ou [CVAT](https://cvat.ai).

### Active learning

Logguer les détections en prod avec `conf < 0.6` pour réannotation humaine → boucle d'amélioration continue.

---

## ✅ Bilan livraison

À la fin de l'exécution complète, tu as :

| Artefact | Chemin | Utilité |
|---|---|---|
| `best.pt` | `/content/kyc_2_1/runs/kyc_2_1_yolov11s/weights/best.pt` | Modèle PyTorch entraîné |
| `best.onnx` | idem `.onnx` | Modèle déployable CPU/edge |
| `predict_and_crop()` | dans ce notebook | Fonction d'interface pour étape 2.2 / 3 |
| `validate_detection_quality()` | dans ce notebook | Règles métier KYC |
| `confusion_matrix.png`, `results.png`, `PR_curve.png` | `runs/.../` | Plots pour le rapport |
| `data.yaml` + dataset YOLO | `/content/kyc_2_1/yolo/` | Réutilisable pour future itération |

**Pour la soutenance :** voir les 3 points à marteler en fin de `plan_technique_kyc_etape_2_1.md`.
